# RisingBALLER MPP — Main Training (Google Colab / Kaggle)

Full Transformer model for Masked Player Prediction pre-training.

**Architecture:** Transformer with self-attention, learned positional encoding (seq_len=36).

**Model config:**
- `embed_size=128`, `num_layers=1`, `heads=2`, `forward_expansion=4`

**Training:** pre-collated batches (batch=256, repeat=10 augmentations), 2000 epochs, lr=1e-4.

**Instructions:**
1. Upload `data_with_dates.csv` to your environment
2. Clone the repo or upload project files
3. Enable GPU in Runtime settings
4. Run all cells

In [ ]:
import os
from pathlib import Path
import shutil


KAGGLE_INPUT = Path("/kaggle/input")
REPO_DIR = Path("/kaggle/working/ML_project-football-")
DATA_CSV = None
for f in KAGGLE_INPUT.rglob("data_with_dates.csv"):
    DATA_CSV = f
    break
if DATA_CSV is None:
    raise FileNotFoundError("data_with_dates.csv not found in /kaggle/input")
REPO_URL = "https://github.com/GibaDuliya/ML_project-football-.git"  # ← CHANGE THIS
BRANCH = "baselines"
if not REPO_DIR.exists():
    import subprocess
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

dest = REPO_DIR / "dataset" / "data_with_dates.csv"
dest.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(DATA_CSV, dest)
DATA_CSV = dest

REPO_DIR = Path("/kaggle/working/ML_project-football-")
DATA_CSV = REPO_DIR / "dataset" / "data_with_dates.csv"
ROOT = REPO_DIR
print("ROOT:", ROOT)
print("DATA_CSV:", DATA_CSV)

import shutil
import torch
import pandas as pd
import numpy as np
from pathlib import Path

# Copy data into repo and create working dirs
dataset_dir = ROOT / "dataset"
dataset_dir.mkdir(parents=True, exist_ok=True)
dest_csv = dataset_dir / "data_with_dates.csv"
if DATA_CSV.resolve() != dest_csv.resolve():
    shutil.copy(DATA_CSV, dest_csv)
    print("Copied to:", dest_csv)

(ROOT / "notebooks" / "RisingBaller" / "mpp_output").mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Data & Vocab

In [ ]:
import os
os.chdir(ROOT)

from data.preprocessing import preprocess_raw_csv, build_vocab_mappings
from data.dataset import MatchDatasetMPP, PreCollatedDataset
from data.collator import DataCollatorMPP
try:
    from data.collator import DataCollatorPreCollated
except ImportError:
    from torch.utils.data.dataloader import default_collate
    class DataCollatorPreCollated:
        def __call__(self, batch):
            return default_collate(batch)
from models.transformer.pretrain import MaskedPlayerModel
from training.trainer import build_training_args, build_trainer
from training.metrics import compute_metrics_mpp

sample_path = ROOT / "notebooks" / "train_sample_raw.csv"
output_dir = str(ROOT / "notebooks" / "train_sample_processed")
Path(output_dir).mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(dest_csv)
df_raw.to_csv(sample_path, index=False)
df = preprocess_raw_csv(str(sample_path), output_dir)
vocab = build_vocab_mappings(df, output_dir)

print(f"Matches: {df['match_id'].nunique()}")
print(f"players_vocab_size (pad+1): {vocab['player_pad_token_id'] + 1}")

## 2. Dataset & Pre-collated Batches

from training.callbacks import BestNCheckpointCallback

model = MaskedPlayerModel(
    embed_size=128,
    num_layers=1,
    heads=2,
    forward_expansion=4,
    dropout=0.05,
    form_stats_size=39,
    players_vocab_size=vocab["player_pad_token_id"],
    teams_vocab_size=vocab["team_pad_token_id"],
    positions_vocab_size=25,
    use_teams_embeddings=False,
    position_enc_type="learned",
)
model = model.to(device)

output_dir = str(ROOT / "notebooks" / "RisingBaller" / "mpp_output")
training_config = {
    "output_dir": output_dir,
    "num_train_epochs": 2000,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "learning_rate": 1e-4,
    "weight_decay": 0.0,
    "warmup_ratio": 0.0,
    "lr_scheduler_type": "linear",
    "logging_steps": 1000,
    "eval_strategy": "steps",
    "eval_steps": 1000,
    "save_strategy": "no",
    "report_to": "none",
    "seed": seed,
}

best_ckpt_cb = BestNCheckpointCallback(
    metrics={"eval_loss": "min", "eval_accuracy_top3": "max"},
    top_n=2,
)

train_args = build_training_args(training_config)
trainer = build_trainer(
    model=model,
    args=train_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics_mpp,
    data_collator=collator_for_trainer,
)
trainer.add_callback(best_ckpt_cb)
best_ckpt_cb.set_trainer(trainer)

print("Parameters:", sum(p.numel() for p in model.parameters()))

final_dir = ROOT / "notebooks" / "RisingBaller" / "mpp_output" / "final_model"
try:
    trainer.train()
finally:
    trainer.save_model(str(final_dir))
    print("Model saved:", final_dir)

    print("\n" + best_ckpt_cb.summary())

    if trainer.state.log_history:
        pd.DataFrame(trainer.state.log_history).to_csv(
            ROOT / "notebooks" / "RisingBaller" / "mpp_output" / "metrics.csv", index=False
        )
        print("\nMetrics saved to metrics.csv")

In [ ]:
import matplotlib.pyplot as plt

history = pd.DataFrame(trainer.state.log_history)
train_logs = history[history["loss"].notna()]
eval_logs = history[history["eval_loss"].notna()]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(train_logs["step"], train_logs["loss"], label="train")
if len(eval_logs) > 0:
    axes[0].plot(eval_logs["step"], eval_logs["eval_loss"], label="eval")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss")
axes[0].legend()

if "eval_accuracy_top1" in eval_logs.columns:
    axes[1].plot(eval_logs["step"], eval_logs["eval_accuracy_top1"], label="top1")
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_title("Top-1 Accuracy")
    axes[1].legend()

if "eval_accuracy_top3" in eval_logs.columns:
    axes[2].plot(eval_logs["step"], eval_logs["eval_accuracy_top3"], label="top3")
    axes[2].set_xlabel("Step")
    axes[2].set_ylabel("Accuracy")
    axes[2].set_title("Top-3 Accuracy")
    axes[2].legend()

plt.suptitle("RisingBALLER MPP Training", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)
results = trainer.evaluate()
print("Eval results:")
for k, v in results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

## 5. Evaluate

In [ ]:
# Запуск обучения (логика из run.py)
import os
os.chdir(REPO_DIR)

import pandas as pd
import torch
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader

from data.preprocessing import preprocess_raw_csv, build_vocab_mappings
from data.dataset import MatchDatasetMPP, PreCollatedDataset
from data.collator import DataCollatorMPP
try:
    from data.collator import DataCollatorPreCollated
except ImportError:
    from torch.utils.data.dataloader import default_collate
    class DataCollatorPreCollated:
        def __call__(self, batch):
            return default_collate(batch)
from models.pretrain import MaskedPlayerModel
from training.trainer import build_training_args, build_trainer
from training.metrics import compute_metrics_mpp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

sample_path = REPO_DIR / "notebooks" / "train_sample_raw.csv"
output_dir = str(REPO_DIR / "notebooks" / "train_sample_processed")
Path(output_dir).mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(dest_csv)
df_raw.to_csv(sample_path, index=False)
df = preprocess_raw_csv(str(sample_path), output_dir)
vocab = build_vocab_mappings(df, output_dir)

print("Матчей:", df["match_id"].nunique(), "players_vocab (pad+1):", vocab["player_pad_token_id"] + 1)

In [ ]:
max_seq_length = 36
sample_batch_size = 256
repeat = 10
dev_ratio = 0.05
seed = 42

# ---- Split by match_id BEFORE creating batches (no data leakage) ----
match_ids = df["match_id"].unique()
np.random.seed(seed)
np.random.shuffle(match_ids)
dev_size = max(1, int(len(match_ids) * dev_ratio))
dev_match_ids = set(match_ids[:dev_size])
train_match_ids = set(match_ids[dev_size:])

df_train = df[df["match_id"].isin(train_match_ids)].reset_index(drop=True)
df_dev = df[df["match_id"].isin(dev_match_ids)].reset_index(drop=True)

print(f"Matches total: {len(match_ids)}, train: {len(train_match_ids)}, dev: {len(dev_match_ids)}")

# ---- Create separate datasets for train and dev ----
ds_train = MatchDatasetMPP(
    df_train,
    player_name2id=vocab["player_name2id"],
    team_name2id=vocab["team_name2id"],
    max_seq_length=max_seq_length,
    player_pad_token_id=vocab["player_pad_token_id"],
    team_pad_token_id=vocab["team_pad_token_id"],
    position_pad_token_id=25,
)
ds_dev = MatchDatasetMPP(
    df_dev,
    player_name2id=vocab["player_name2id"],
    team_name2id=vocab["team_name2id"],
    max_seq_length=max_seq_length,
    player_pad_token_id=vocab["player_pad_token_id"],
    team_pad_token_id=vocab["team_pad_token_id"],
    position_pad_token_id=25,
)

collator = DataCollatorMPP(
    player_mask_token_id=vocab["player_mask_token_id"],
    mask_percentage=0.25,
)

def _collate_filter_none(batch):
    batch = [b for b in batch if b is not None]
    return collator(batch) if batch else None

# ---- Build train batches ----
loader_train = DataLoader(
    ds_train, batch_size=sample_batch_size, shuffle=True,
    collate_fn=_collate_filter_none, drop_last=True,
)
train_batches = []
for _ in range(repeat):
    for batch in loader_train:
        if batch is not None:
            train_batches.append(batch)

# ---- Build dev batches (drop_last=False to keep all dev matches) ----
loader_dev = DataLoader(
    ds_dev, batch_size=sample_batch_size, shuffle=True,
    collate_fn=_collate_filter_none, drop_last=False,
)
dev_batches = []
for _ in range(repeat):
    for batch in loader_dev:
        if batch is not None:
            dev_batches.append(batch)

train_dataset = PreCollatedDataset(train_batches)
eval_dataset = PreCollatedDataset(dev_batches)
collator_for_trainer = DataCollatorPreCollated()

print(f"Train batches: {len(train_batches)}, Dev batches: {len(dev_batches)}")

In [ ]:
from training.callbacks import BestNCheckpointCallback

model = MaskedPlayerModel(
    embed_size=128,
    num_layers=1,
    heads=2,
    forward_expansion=4,
    dropout=0.05,
    form_stats_size=39,
    players_vocab_size=vocab["player_pad_token_id"],
    teams_vocab_size=vocab["team_pad_token_id"],
    positions_vocab_size=25,
    use_teams_embeddings=False,
    position_enc_type="learned",
)
model = model.to(device)

output_dir = str(REPO_DIR / "notebooks" / "mpp_mini_output")
training_config = {
    "output_dir": output_dir,
    "num_train_epochs": 2000,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "learning_rate": 1e-4,
    "weight_decay": 0.0,
    "warmup_ratio": 0.0,
    "lr_scheduler_type": "linear",
    "logging_steps": 1000,
    "eval_strategy": "steps",
    "eval_steps": 1000,
    "save_strategy": "no",
    "report_to": "none",
    "seed": seed,
}

best_ckpt_cb = BestNCheckpointCallback(
    metrics={"eval_loss": "min", "eval_accuracy_top3": "max"},
    top_n=2,
)

train_args = build_training_args(training_config)
trainer = build_trainer(
    model=model,
    args=train_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics_mpp,
    data_collator=collator_for_trainer,
)
trainer.add_callback(best_ckpt_cb)
best_ckpt_cb.set_trainer(trainer)

print("Параметров:", sum(p.numel() for p in model.parameters()))

In [ ]:
try:
    trainer.train()
finally:
    # Save final model (last state)
    final_dir = REPO_DIR / "notebooks" / "mpp_mini_output" / "final_model"
    trainer.save_model(str(final_dir))
    print("Финальная модель сохранена:", final_dir)

    # Print best checkpoints summary
    print("\n" + best_ckpt_cb.summary())

    # Save metrics history
    if trainer.state.log_history:
        pd.DataFrame(trainer.state.log_history).to_csv(
            REPO_DIR / "notebooks" / "mpp_mini_output" / "metrics.csv", index=False
        )
        print("\nМетрики сохранены в metrics.csv")